In [1]:
#pip install -U ultralytics

In [1]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

#config
VIDEO_PATH = Path("20250813_Group6_0650-0750_15min.mp4")
MODEL_PATH = Path("runs/detect/dogs_nano/weights/best.pt")  # or dogs_medium/weights/best.pt
OUTPUT_DIR = Path("runs/tracking_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRACKER_CONFIG = "custom_botsort.yaml"  # Built-in Ultralytics BoT-SORT- now custom
CONF_THRESHOLD = 0.35            # Minimum detection confidence
MOVING_AVG_WINDOW = 5            # Frames for smoothing speed jitter
PROXIMITY_THRESHOLD_PX = 80.0    # Distance in pixels considered an interaction



In [2]:
#tracking inference
print(f"Loading model: {MODEL_PATH}")
model = YOLO(str(MODEL_PATH))

print(f"Processing video: {VIDEO_PATH}")
cap = cv2.VideoCapture(str(VIDEO_PATH))
fps = cap.get(cv2.CAP_PROP_FPS) or 49.95
dt = 1.0 / fps
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

print(f"Video Info: {total_frames} frames | {fps:.2f} FPS | Resolution: {width}x{height}")

# Stream tracking results directly
results = model.track(
    source=str(VIDEO_PATH),
    tracker=TRACKER_CONFIG,
    conf=CONF_THRESHOLD,
    persist=True,
    stream=True,
    verbose=False
)

raw_records = []
frame_idx = 0

print("Running detection and trajectory association...")
for r in results:
    frame_idx += 1
    timestamp = frame_idx * dt
    
    if r.boxes is not None and r.boxes.id is not None:
        boxes_xywh = r.boxes.xywh.cpu().numpy()  # [cx, cy, w, h]
        track_ids = r.boxes.id.int().cpu().numpy()
        confs = r.boxes.conf.cpu().numpy()
        
        for box, track_id, conf in zip(boxes_xywh, track_ids, confs):
            cx, cy, w, h = box
            raw_records.append({
                "frame": frame_idx,
                "timestamp_sec": round(timestamp, 3),
                "track_id": int(track_id),
                "center_x": round(float(cx), 2),
                "center_y": round(float(cy), 2),
                "box_width": round(float(w), 2),
                "box_height": round(float(h), 2),
                "confidence": round(float(conf), 4)
            })

df_raw = pd.DataFrame(raw_records)
print(f"Tracking complete. Extracted {len(df_raw)} detections across {frame_idx} frames.")

Loading model: runs/detect/dogs_nano/weights/best.pt
Processing video: 20250813_Group6_0650-0750_15min.mp4
Video Info: 44954 frames | 49.95 FPS | Resolution: 720x576
Running detection and trajectory association...
Tracking complete. Extracted 122014 detections across 44954 frames.


In [3]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [4]:
#KINEMATIC COMPUTATIONS (SPEED, DISTANCE, PROXIMITY)

print("Computing kinematic metrics...")
processed_dfs = []

# Process each unique tracked animal independently
for track_id, group in df_raw.groupby("track_id"):
    group = group.sort_values("frame").copy()
    
    # Calculate Euclidean step displacement between consecutive frames
    dx = group["center_x"].diff().fillna(0)
    dy = group["center_y"].diff().fillna(0)
    step_dist = np.sqrt(dx**2 + dy**2)
    
    # Cumulative distance
    group["step_distance_px"] = step_dist
    group["cumulative_dist_px"] = step_dist.cumsum()
    
    # Instantaneous speed (pixels/sec)
    group["speed_px_per_sec"] = step_dist / dt
    
    # Smoothed speed using moving average filter
    group["speed_smoothed_px_per_sec"] = (
        group["speed_px_per_sec"]
        .rolling(window=MOVING_AVG_WINDOW, center=True, min_periods=1)
        .mean()
    )
    
    processed_dfs.append(group)

df_trajectories = pd.concat(processed_dfs, ignore_index=True)

# Save per-frame detailed trajectory CSV
traj_csv_path = OUTPUT_DIR / "trajectories_per_frame.csv"
df_trajectories.to_csv(traj_csv_path, index=False)
print(f"Saved detailed trajectories to: {traj_csv_path}")



Computing kinematic metrics...
Saved detailed trajectories to: runs/tracking_results/trajectories_per_frame.csv


In [5]:
#summary stats per dogs
summary_records = []
for track_id, group in df_trajectories.groupby("track_id"):
    total_dist = group["step_distance_px"].sum()
    mean_speed = group["speed_smoothed_px_per_sec"].mean()
    max_speed = group["speed_smoothed_px_per_sec"].max()
    active_frames = (group["speed_smoothed_px_per_sec"] > 20.0).sum()
    
    summary_records.append({
        "track_id": track_id,
        "total_frames_tracked": len(group),
        "tracking_duration_sec": round(len(group) * dt, 2),
        "total_distance_px": round(total_dist, 2),
        "mean_speed_px_s": round(mean_speed, 2),
        "max_speed_px_s": round(max_speed, 2),
        "locomotion_time_pct": round((active_frames / len(group)) * 100, 2)
    })

df_summary = pd.DataFrame(summary_records).sort_values("track_id")
summary_csv_path = OUTPUT_DIR / "summary_metrics_per_dog.csv"
df_summary.to_csv(summary_csv_path, index=False)
print(f"Saved summary metrics to: {summary_csv_path}")



Saved summary metrics to: runs/tracking_results/summary_metrics_per_dog.csv


In [6]:
#visualize

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

# Plot 1: 2D Spatial Trajectory Map
fig, ax = plt.subplots(figsize=(10, 8))
for track_id, group in df_trajectories.groupby("track_id"):
    if len(group) > 50:  # Plot prominent tracks
        ax.plot(group["center_x"], group["center_y"], label=f"Dog (ID {track_id})", alpha=0.7, linewidth=1.5)
        ax.scatter(group["center_x"].iloc[0], group["center_y"].iloc[0], marker="o", s=40)
        ax.scatter(group["center_x"].iloc[-1], group["center_y"].iloc[-1], marker="x", s=40)

ax.set_xlim(0, width)
ax.set_ylim(height, 0)  # Invert Y to match video pixel coordinates
ax.set_title("Canine 2D Spatial Trajectories (Full Session)", fontsize=13, fontweight="bold")
ax.set_xlabel("X coordinate (pixels)")
ax.set_ylabel("Y coordinate (pixels)")
ax.legend(loc="upper right", frameon=True)
plt.tight_layout()
traj_plot_path = OUTPUT_DIR / "spatial_trajectories_plot.png"
plt.savefig(traj_plot_path, dpi=150)
plt.close()

# Plot 2: Speed Over Time
fig, ax = plt.subplots(figsize=(12, 5))
for track_id, group in df_trajectories.groupby("track_id"):
    if len(group) > 100:
        ax.plot(group["timestamp_sec"], group["speed_smoothed_px_per_sec"], label=f"ID {track_id}", alpha=0.8)

ax.set_title("Smoothed Locomotion Velocity over Time", fontsize=13, fontweight="bold")
ax.set_xlabel("Time (seconds)")
ax.set_ylabel("Speed (pixels / second)")
ax.legend(loc="upper right", frameon=True)
plt.tight_layout()
speed_plot_path = OUTPUT_DIR / "velocity_profiles_plot.png"
plt.savefig(speed_plot_path, dpi=150)
plt.close()

print("All tracking artifacts and plots generated successfully!")

/tmp/ipykernel_1599763/1071115842.py:19: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
/tmp/ipykernel_1599763/1071115842.py:34: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


All tracking artifacts and plots generated successfully!
